In [ ]:
import arya
import sys
import matplotlib.pyplot as plt

In [ ]:
sys.path.append("../imaging/")
import convenience_functions

In [ ]:
from astropy.nddata import CCDData

In [ ]:
import numpy as np

In [ ]:
from reproject import reproject_interp


In [ ]:
import tomllib

In [ ]:
import sys
sys.path.append("../photometry/")

In [ ]:
from phot_utils import get_atm_extinction, to_mag, show_image


In [ ]:
yasone1_bkg_err = 40
yasone2_bkg_err = 40
yasone3_bkg_err = 40

In [ ]:
def get_overlapping_regions(img, img2, xshift=0, yshift=0):
    h1 = img.wcs.to_header()
    h2 = img2.wcs.to_header()
    
    shape1 = img.header["naxis1"], img.header["naxis2"]
    shape2 = img2.header["naxis1"], img2.header["naxis2"]
    
    coord = img2.wcs.pixel_to_world((shape2[0]-1)/ 2, (shape2[1]-1)/ 2)
    if isinstance(coord, list):
        x0, y0 = img.wcs.world_to_pixel(*coord)

    else:
        x0, y0 = img.wcs.world_to_pixel(coord)

    dx = h2["crpix1"] - h1["crpix1"] + (shape2[1] - shape1[1])/2
    dy = h2["crpix2"] - h1["crpix2"]
    assert 0 == (h1["CRVAL1"] - h2["CRVAL1"])
    assert 0 == (h1["CRVAL2"] - h2["CRVAL2"])



    print(x0, y0)
    print(dx, dy)
    print(dx + np.ceil(shape1[0]/2), dy + np.ceil(shape1[1]/2))
    print(shape1)
    print(shape2)
    # print(h2["crpix1"]/shape2[0])

    # print(h2["crpix1"]/shape2[0])

    idx, idx2 = overlap_slices(shape1, shape2, (x0+xshift, y0 + yshift)) # centering seems to shift by one pixel...
    # print(idx, idx2)
    return img.data[idx[1], idx[0]], img2.data[idx2[1], idx2[0]] # fits order...
    

In [ ]:
def clip_image(img, vmin=None, vmax=None):
    img_clipped = img.copy()
    for i in range(3):
        if vmin is None:
            l = np.nanmin(img[:, :, i])
        else:
            l = vmin
        if vmax is None:
            h = np.nanmax(img[:, :, i])
        else:
            h = vmax
        img_clipped[:, :, i] = (img[:, :, i] - l) / (h-l)

    return np.maximum(np.minimum(img_clipped, 1), 0)

In [ ]:
def color_clip_image(img, vmin=0, vmax=None, norm=np.asinh, scale=1):
    img_clipped = img.copy()
    I = np.sum(img, axis=2) / 3
    if vmax is None:
        h = np.quantile(I[np.isfinite(I)], 0.99)
    else:
        h = vmax
    l = vmin

    def f(I):
        x = (norm(I/scale) - norm(l/scale)) / (norm(h/scale) - norm(l/scale))
        return x
    for i in range(3):
        img_clipped[:, :, i] *= f(I) / I

    return img_clipped

In [ ]:
def combine_images(img_g, img_r, img_i):
    img_i_interp, footprint_i = reproject_interp(img_i, img_r.wcs, shape_out=img_r.shape)
    img_g_interp, footprint_g = reproject_interp(img_g, img_r.wcs, shape_out=img_r.shape)
    
    img = np.stack([
            img_i_interp,
            img_r.data,
            img_g_interp
        ], axis=2
    )

    return img
    

In [ ]:
def combine_flag_images(img_g, img_r, img_i):
    img_i_interp, footprint_i = reproject_interp(img_i, img_r.wcs, shape_out=img_r.shape, order=0)
    img_i_interp = img_i_interp.astype(img_r.data.dtype)
    img_g_interp, footprint_g = reproject_interp(img_g, img_r.wcs, shape_out=img_r.shape, order=0)
    img_g_interp = img_g_interp.astype(img_r.data.dtype)
    
    img = img_i_interp | img_r.data |  img_g_interp
    return img
    

In [ ]:
def get_zeropoint(obj, filt, exposure=190, gain=1.9, airmass=1.2):
    d = None
    with open(f"../photometry/std1/img_{filt}_11/flat_fielded-astrom-zeropoint.toml", "rb") as f:
        d = tomllib.load(f)

    return d["zeropoint"] + d["ap_corr"] + 2.5 * np.log10(exposure) - get_atm_extinction(airmass, f"Sloan_{filt}")[0] - 2.5 * np.log10(gain) - get_mag_shift(obj)[filt]

In [ ]:
def to_flux_image(img, objname):
    zp_g = get_zeropoint(objname, "g")
    zp_r = get_zeropoint(objname, "r")
    zp_i = get_zeropoint(objname, "i")

    img_flux = img * 10 ** (-0.4 * np.array([zp_i, zp_r, zp_g]).reshape(1, 1, -1))

    return img_flux

In [ ]:
def to_mag_image(img, objname):
    img_mag = -2.5 * np.log10(img)
    zp_g = get_zeropoint(objname, "g")
    zp_r = get_zeropoint(objname, "r")
    zp_i = get_zeropoint(objname, "i")

    img_mag += np.array([zp_i, zp_r, zp_g]).reshape(1, 1, -1)

    return img_mag

In [ ]:
from astropy.nddata import block_reduce
def show_image(image,
               figsize=(10, 10),
               dpi = None,
               clabel=None,
               clim=None,
               show_colorbar=True, show_ticks=True,
               fig=None, ax=None, input_ratio=None,
              subplot_kw=dict()):
    """
    Show an image in matplotlib with some basic astronomically-appropriate stretching.

    Parameters
    ----------
    image
        The image to show
    percl : number
        The percentile for the lower edge of the stretch (or both edges if ``percu`` is None)
    percu : number or None
        The percentile for the upper edge of the stretch (or None to use ``percl`` for both)
    figsize : 2-tuple
        The size of the matplotlib figure in inches
    """

    if dpi is None:
        dpi = image.shape[0] / figsize[0]

    if (fig is None and ax is not None) or (fig is not None and ax is None):
        raise ValueError('Must provide both "fig" and "ax" '
                         'if you provide one of them')
    elif fig is None and ax is None:
        if figsize is not None:
            # Rescale the fig size to match the image dimensions, roughly
            image_aspect_ratio = image.shape[0] / image.shape[1]
            figsize = (max(figsize) * image_aspect_ratio, max(figsize))

        fig, ax = plt.subplots(1, 1, figsize=figsize, dpi=dpi, subplot_kw=subplot_kw)


    # To preserve details we should *really* downsample correctly and
    # not rely on matplotlib to do it correctly for us (it won't).

    # So, calculate the size of the figure in pixels, block_reduce to
    # roughly that,and display the block reduced image.

    # Thanks, https://stackoverflow.com/questions/29702424/how-to-get-matplotlib-figure-size

    fig_size_pix = fig.get_size_inches() * fig.dpi

    ratio = (image.shape[0:2] // fig_size_pix).max()

    if ratio < 1:
        ratio = 1

    ratio = input_ratio or ratio

    reduced_data = block_reduce(image, ratio)

    # Divide by the square of the ratio to keep the flux the same in the
    # reduced image. We do *not* want to do this for images which are
    # masks, since their values should be zero or one.
    reduced_data = reduced_data / ratio**2

    # Of course, now that we have downsampled, the axis limits are changed to
    # match the smaller image size. Setting the extent will do the trick to
    # change the axis display back to showing the actual extent of the image.
    extent = [0, image.shape[1], 0, image.shape[0]]


    im = ax.imshow(reduced_data, origin='lower',
                   extent=extent, aspect='equal')


    if not show_ticks:
        ax.tick_params(labelbottom=False, labelleft=False, labelright=False, labeltop=False)


In [ ]:
from ana_utils import get_mag_shift

In [ ]:
get_zeropoint("yasone1", "i")

In [ ]:
import ana_utils

In [ ]:
def plot_nice_image(img_mag, img_flag, wcs):
    
    fig, ax = plt.subplots(
        dpi = 1000,
        subplot_kw = dict(projection=wcs)
    )

    plt.xlabel(r"$\alpha$ / degree")
    plt.ylabel(r"$\delta$ / degree")
    
    clipped = clip_image(-img_mag, vmin=-30)
    clipped[np.isnan(clipped)] = 0
    clipped[img_flag & (8) > 0] = 1
    plt.imshow(clipped )


# Yasone 1

In [ ]:
def ra_dec_axis():
    plt.xlabel(r"Right Ascension")
    plt.ylabel(r"Declination")

In [ ]:
ra_dec_axis()

In [ ]:
yasone1 = CCDData.read("../photometry/yasone1/coadd_median_r/coadd.fits", unit="adu")
yasone1_g = CCDData.read("../photometry/yasone1/coadd_median_g/coadd.fits", unit="adu")
yasone1_i = CCDData.read("../photometry/yasone1/coadd_median_i/coadd.fits", unit="adu")



yasone1_flags = CCDData.read("../photometry/yasone1/coadd_median_r/flag.fits", unit="adu")
yasone1_g_flags = CCDData.read("../photometry/yasone1/coadd_median_g/flag.fits", unit="adu")
yasone1_i_flags = CCDData.read("../photometry/yasone1/coadd_median_i/flag.fits", unit="adu")


In [ ]:
img_raw = combine_images(yasone1_g, yasone1, yasone1_i)

In [ ]:
img_flag = combine_flag_images(yasone1_flags, yasone1_g_flags, yasone1_i_flags)


In [ ]:
img_mag = to_mag_image(img_raw, "yasone1")

In [ ]:
img_flux = to_flux_image(img_raw, "yasone1")

In [ ]:
plt.imshow(img_flag & (4 + 8))

In [ ]:
from astropy.visualization import make_rgb, ManualInterval, LogStretch, make_lupton_rgb


In [ ]:
np.sort(np.unique(10**-img_mag[:, :, 0]))

In [ ]:
z0 = 0.4*26 
img_c = make_lupton_rgb(10**(z0-0.4*img_mag[:, :, 0]), 10**(z0-0.4*img_mag[:, :, 1]), 10**(z0-0.4*img_mag[:, :, 2]),
                        Q = 10, stretch=1
                       )
plt.imshow(img_c, origin="lower")

In [ ]:
z0 = 0.4*25
img_c = make_lupton_rgb(img_flux[:, :, 0], img_flux[:, :, 1], img_flux[:, :, 2],
                        Q = 1, stretch=10**-z0
                       )
plt.imshow(img_c, origin="lower")

In [ ]:
clipped = color_clip_image(img_flux, scale=10e-12, vmax=1e-9,)
clipped[~np.isfinite(clipped)] = 0
plt.imshow(clipped, origin="lower")

In [ ]:
plt.hist(np.log10(img_flux[img_flux > 0]))

In [ ]:
clipped = color_clip_image(img_flux, scale=10e-12, vmax=1e-10,)
clipped[~np.isfinite(clipped)] = 0
show_image(clipped, subplot_kw=dict(projection=yasone1.wcs))
ra_dec_axis()

plt.savefig("figures/yasone1_photo.jpg")

In [ ]:
clipped = color_clip_image(img_flux, vmax=1e-10, norm=lambda x: x)
clipped[~np.isfinite(clipped)] = 0
show_image(clipped, subplot_kw=dict(projection=yasone1.wcs))

In [ ]:
clipped = color_clip_image(img_flux, scale=5e-14, vmin=1e-12, vmax=1e-10, norm=np.log10)
clipped[~np.isfinite(clipped)] = 0
show_image(clipped, subplot_kw=dict(projection=yasone1.wcs))

In [ ]:
plt.hist(np.log10(img_flux[img_flux > 0]))

In [ ]:
clipped = color_clip_image(img_flux, vmax=1e-10, vmin=1e-16, norm=np.log10)

In [ ]:
plt.hist(clipped[np.isfinite(clipped)], bins=np.linspace(-1, 2, 100))

In [ ]:
plt.figure(dpi=200)
clipped[np.isnan(clipped)] = 0
plt.imshow(clipped, origin="lower")

In [ ]:
plot_nice_image(img_mag, img_flag, yasone1.wcs)

In [ ]:
np.moveaxis(img_mag, -1, 0)

In [ ]:
CCDData(np.moveaxis(img_mag, -1, 0), unit="adu").write("pretty_yasone1.fits", overwrite=True)

## Yasone 2

In [ ]:

yasone2 = CCDData.read("../photometry/yasone2/coadd_median_r/coadd.fits", unit="adu")
yasone2_g = CCDData.read("../photometry/yasone2/coadd_median_g/coadd.fits", unit="adu")
yasone2_i = CCDData.read("../photometry/yasone2/coadd_median_i/coadd.fits", unit="adu")

yasone3 = CCDData.read("../photometry/yasone3/coadd_median_r/coadd.fits", unit="adu")
yasone3_g = CCDData.read("../photometry/yasone3/coadd_median_g/coadd.fits", unit="adu")
yasone3_i = CCDData.read("../photometry/yasone3/coadd_median_i/coadd.fits", unit="adu")

In [ ]:
yasone2_flags = CCDData.read("../photometry/yasone2/coadd_median_r/flag.fits", unit="adu")
yasone2_g_flags = CCDData.read("../photometry/yasone2/coadd_median_g/flag.fits", unit="adu")
yasone2_i_flags = CCDData.read("../photometry/yasone2/coadd_median_i/flag.fits", unit="adu")


In [ ]:
img_raw = combine_images(yasone2_g, yasone2, yasone2_i)


In [ ]:
img_flag = combine_flag_images(yasone2_g_flags, yasone2_flags, yasone2_i_flags)

In [ ]:
img_flux = to_flux_image(img_raw, "yasone2")

In [ ]:
clipped = color_clip_image(img_flux, scale=5e-12, vmin=0, vmax=1e-10)

In [ ]:
clipped[~np.isfinite(clipped)] = 0
show_image(clipped, subplot_kw=dict(projection=yasone2.wcs))
ra_dec_axis()
plt.savefig("figures/yasone2_photo.jpg")

## Yasone 3

In [ ]:
yasone3_flags = CCDData.read("../photometry/yasone3/coadd_median_r/flag.fits", unit="adu")
yasone3_g_flags = CCDData.read("../photometry/yasone3/coadd_median_g/flag.fits", unit="adu")
yasone3_i_flags = CCDData.read("../photometry/yasone3/coadd_median_i/flag.fits", unit="adu")


In [ ]:
img_raw = combine_images(yasone3_g, yasone3, yasone3_i)


In [ ]:
img_flag = combine_flag_images(yasone3_g_flags, yasone3_flags, yasone3_i_flags)

In [ ]:
img_flux = to_flux_image(img_raw, "yasone3")

In [ ]:
plt.hist(img_flux[img_flux > 0])

In [ ]:
clipped = color_clip_image(img_flux, scale=6e-12, vmin=0, vmax=1e-10)
clipped[~np.isfinite(clipped)] = 0

show_image(clipped, subplot_kw=dict(projection=yasone3.wcs))
ra_dec_axis()
plt.savefig("figures/yasone3_photo.jpg")

In [ ]:
plt.imshow(img_flag & (4))

In [ ]:
img_mag = to_mag_image(img_raw, "yasone3")

In [ ]:
plot_nice_image(img_mag, img_flag, yasone2.wcs)

# Single filter

In [ ]:
cmap = plt.get_cmap("mako")
cmap.set_bad("k")

In [ ]:
cmap

In [ ]:
fig, ax = plt.subplots(
    dpi = 200,
    subplot_kw = dict(projection=yasone1.wcs)
)
p = ax.imshow(yasone1.data, vmin=yasone1_bkg_err, cmap=cmap, norm="log")
plt.colorbar(p)

In [ ]:
fig, ax = plt.subplots(
    dpi = 200,
    subplot_kw = dict(projection=yasone1.wcs)
)
p = ax.imshow(yasone2.data, vmin=yasone2_bkg_err, cmap=cmap, norm="log")
plt.colorbar(p)

In [ ]:
fig, ax = plt.subplots(
    dpi = 200,
    subplot_kw = dict(projection=yasone1.wcs)
)
p = ax.imshow(yasone3.data, vmin=yasone3_bkg_err, cmap=cmap, norm="log")
plt.colorbar(p)